In [10]:
%matplotlib widget
# Para manipulación y análisis de datos
!pip install pandas pymongo

# Para visualización de datos (gráficas)
!pip install matplotlib seaborn

# Opcional: Para análisis de sentimiento y lenguaje (muy cool para Andrés)
!pip install textblob

# Instalamos la versión CPU para ahorrar espacio
!pip install torch --index-url https://download.pytorch.org/whl/cpu
# Luego el resto
!pip install sentence-transformers scikit-learn scipy

!pip install ipympl

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 14.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 87.5 MB/s eta 0:00:00


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from pymongo import MongoClient
import pandas as pd
import seaborn as sns
import numpy as np
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Obtención de datos
Se toman los datos obtenidos y guardados en la base de datos con el fin de iniciar el análisis

In [3]:
# Conexión al servicio de MongoDB
client = MongoClient('mongodb://mongodb:27017/')
db = client['social_analytics']
collections = db.list_collection_names()
print("🗄️ Colecciones encontradas en 'social_analytics':")
for c in collections:
    # Contamos cuántos documentos hay en cada una para darle más info a Andrés
    count = db[c].count_documents({})
    print(f" - {c} ({count} documentos)")



🗄️ Colecciones encontradas en 'social_analytics':
 - tweets (3250 documentos)


In [4]:
collection = db['tweets']
# Convertir la colección a DataFrame
df = pd.DataFrame(list(collection.find()))

# Verificar si cargó algo
if df.empty:
    print("⚠️ El DataFrame está vacío. ¡Revisa si el Cron ya guardó algo!")
else:
    print(f"✅ Éxito: {len(df)} registros cargados en 'df'.")

# Ver una muestra de los datos
display(df)

✅ Éxito: 3250 registros cargados en 'df'.


,_id,tweet_id,account_label,author,content,timestamp_detected,timestamp_posted
0,69d6d770d4b5be3196616259,2041660134846558664,Gamer_Project,@penikmat_hujann,What the fuck just happened?,2026-04-08T22:32:16.245440,2026-04-08T22:32:16.245427
1,69d6d770d4b5be319661625a,2041795685754818669,marthasteward295,@KoruturkBey,"In Canada, a streamer nicknamed Joker was raid...",2026-04-09T04:15:51.378691,2026-04-09T04:15:51.378666
2,69d6d770d4b5be319661625b,2041802128994275586,marthasteward295,@Kaliza_queen,This video say a lot about nowadays society......,2026-04-09T07:58:02.429904,2026-04-09T07:58:02.429891
3,69d6d770d4b5be319661625c,analytics,animecol300_Project,@Glencore,"At Nikkelverk, Eirik’s job is to find ways to ...",2026-04-09T20:24:44.971681,2026-04-09T20:24:44.971663
4,69d6d770d4b5be319661625d,2041954231817769138,Gamer_Project,@TheNetDaily,Jake Paul Got Caught,2026-04-08T22:51:53.851682,2026-04-08T22:51:53.851668
...,...,...,...,...,...,...,...
3245,69d80b00d68eedba1b900fd0,2042173497984819426,animecol300_Project,@dogsmellsgood,Holy fucking shit. Finally found the name for ...,2026-04-09T20:24:44.878011,2026-04-09T20:24:44.877996
3246,69d80b00d68eedba1b900fd1,2042243841017655605,animecol300_Project,@quesadaaa_,"gel?... gel? our options are patches, shots, i...",2026-04-09T20:24:32.771126,2026-04-09T20:24:32.771113
3247,69d80b0dd68eedba1b900fd2,2042239828272496732,animecol300_Project,@fernechini,que ganas de comer PAPASAURIOS,2026-04-09T20:24:45.081397,2026-04-09T20:24:45.081385
3248,69d80b1dd68eedba1b900fd3,2042329147087708350,marthasteward295,@aldogeotv,"I'm on my way back to CDMX, dads, don't worry,...",2026-04-09T20:24:59.811019,2026-04-09T20:24:59.811001


Se hace una exploración inicial de los datos para identificar cuales son las cuentas de las que más se reciben tweets, se evidencia que a pesar de que ninguna cuenta sigue a @elonmusk se ve que las cuentas reciben tweets de él.

In [5]:
# 1. Filtramos los Top 10 autores
top_10_names = df['author'].value_counts().head(10).index
df_top = df[df['author'].isin(top_10_names)]

# 2. Creamos el pivot
pivot_df = df_top.groupby(['author', 'account_label']).size().unstack(fill_value=0)

# 3. Ordenar por total
pivot_df['total'] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values('total', ascending=True).drop(columns='total')

num_proyectos = len(pivot_df.columns)
colores_proyecto = sns.color_palette("Set2", num_proyectos)

# 4. Graficamos con la nueva paleta
ax = pivot_df.plot(kind='barh', stacked=True, figsize=(12, 8), color=colores_proyecto)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        x = p.get_x() + width / 2
        y = p.get_y() + p.get_height() / 2
        ax.text(x, y, f'{int(width)}', ha='center', va='center', color='white', fontweight='bold')

plt.title('Análisis por Autor: Distribución en los 3 Proyectos', fontsize=15)
plt.xlabel('Cantidad de Tweets', fontsize=12)
plt.ylabel('Autor', fontsize=12)
plt.legend(title='Proyecto', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

<IPython.core.display.Javascript object>

In [6]:
# 1. Preparar los datos
conteo = df['account_label'].value_counts()
total_tweets = conteo.sum()

# 2. Generar los colores manualmente (solución al error TypeError)
# Usamos el mapa de colores 'Pastel1' basado en la cantidad de categorías que tengamos
colores = plt.get_cmap('Pastel1')(np.linspace(0, 1, len(conteo)))

# 3. Función para mostrar porcentaje y cantidad
def fmt_cantidad(pct):
    absolute = int(round(pct/100.*total_tweets))
    return f"{pct:.1f}%\n({absolute} tweets)"

plt.figure(figsize=(10, 7))

# 4. Dibujar la torta (ahora usamos 'colors' en lugar de 'cmap')
plt.pie(conteo, labels=conteo.index, autopct=fmt_cantidad, 
        startangle=90, colors=colores, pctdistance=0.80,
        textprops={'fontsize': 12, 'fontweight': 'bold'})

# 5. Dibujar el círculo blanco central para que sea una DONA
centro_circulo = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centro_circulo)

# 6. Poner el gran TOTAL en el centro de la dona
plt.text(0, 0, f'TOTAL\n{total_tweets}', ha='center', va='center', 
         fontsize=18, fontweight='bold', color='#333333')

plt.title('Análisis de Recolección por Proyecto', fontsize=16, pad=20)
plt.axis('equal') 
plt.tight_layout()
plt.show()

<IPython.core.display.Javascript object>

## Procesamiento de texto a vectores (Embeddings)
Convertir el contenido de los tweets en números.


In [7]:


# 1. Cargamos el modelo (el mismo que citaron en su artículo)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Tomamos los textos de nuestra base de datos (df['content'])
textos = df['content'].fillna("").tolist()

# 3. ¡MAGIA! Convertimos los 164 tweets en vectores
# Cada tweet ahora es una lista de 384 números
embeddings = model.encode(textos)

print(f"✅ Hemos convertido {len(embeddings)} tweets en vectores de dimensión {embeddings.shape[1]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5173.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Hemos convertido 3250 tweets en vectores de dimensión 384


In [8]:
print(embeddings)

[[-0.05179118  0.00033369  0.0511012  ... -0.00082497  0.05301495
   0.01868276]
 [-0.06202882  0.00303204 -0.06478949 ...  0.05405934 -0.01374433
   0.0125442 ]
 [-0.04360653 -0.01821359 -0.10178803 ... -0.01710293  0.11621358
  -0.02902644]
 ...
 [-0.07748342  0.07513889  0.00548675 ...  0.07803216  0.08856342
  -0.00211502]
 [ 0.05630726 -0.0961883   0.04024629 ... -0.06962743 -0.0230589
  -0.05848522]
 [ 0.00981461 -0.02804882  0.03745189 ... -0.00724437 -0.08759533
   0.02230389]]


In [9]:
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

from mpl_toolkits.mplot3d import Axes3D



# 1. Reducir a 3 dimensiones (PCA)

pca_3d = PCA(n_components=3)

coords_3d = pca_3d.fit_transform(embeddings)

df['x_3d'] = coords_3d[:, 0]

df['y_3d'] = coords_3d[:, 1]

df['z_3d'] = coords_3d[:, 2]



# 2. Configurar la figura 3D

fig = plt.figure(figsize=(12, 10))

ax = fig.add_subplot(111, projection='3d')



# 3. Mapear colores por Cluster para mantener la lógica anterior

# (O puedes cambiar 'hue' a 'account_label' si quieres ver los proyectos por color)

categorias = df['account_label'].unique()

colores = ['#5d2a80', '#2a805d', '#ff7f0e'] # Un color por cada cuenta



for i, cuenta in enumerate(categorias):

    subset = df[df['account_label'] == cuenta]

    ax.scatter(subset['x_3d'], subset['y_3d'], subset['z_3d'], 

               label=cuenta, s=60, alpha=0.7)



# 4. Etiquetas de los ejes

ax.set_title(f'Análisis Espacial 3D: {len(df)} Tweets', fontsize=16)

ax.set_xlabel('Componente Principal 1')

ax.set_ylabel('Componente Principal 2')

ax.set_zlabel('Componente Principal 3')



ax.legend(title="Proyectos")

plt.show()

<IPython.core.display.Javascript object>